# Workshop — RAG sobre papers académicos con LangChain

En este workshop vas a construir un sistema RAG completo usando LangChain,
alimentado con dos papers fundamentales de IA:

- **attention_paper.pdf** — "Attention Is All You Need" (Vaswani et al., 2017)
- **rag_paper.pdf**       — "RAG: Retrieval-Augmented Generation" (Lewis et al., 2020)

Al final podrás hacerle preguntas a los papers y el sistema responderá
basándose únicamente en el contenido que hayas indexado.

**Stack:** MarkItDown · LangChain · Qdrant · Gemini 2.5 Flash Lite

**Estructura:**
| Parte | Tarea |
|---|---|
| 1 | Cargar y convertir PDFs con MarkItDown |
| 2 | Convertir a Documents de LangChain y chunkear |
| 3 | Crear embeddings e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar con las preguntas del quiz |

In [1]:
# Dependencias (ya instaladas en el entorno local con requirements.txt)
# !pip install -q markitdown langchain langchain-google-genai langchain-qdrant
# !pip install -q langchain-text-splitters qdrant-client python-dotenv

In [2]:
import os
from pathlib import Path
from typing import List
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, "Falta GOOGLE_API_KEY en .env"

print("Setup completo")

Setup completo


---
## Parte 1 — Cargar PDFs con MarkItDown

MarkItDown convierte cualquier PDF a texto Markdown con una sola línea.
El texto resultante preserva la estructura del documento (secciones, listas)
y es directamente usable como input para el chunker.

**Tu tarea:** completar la función `load_pdf()` y cargar los dos papers.

In [3]:
from markitdown import MarkItDown

md_converter = MarkItDown(enable_plugins=False)

DOCS_DIR = Path("papers")

def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}. "
                         "Verifica que el PDF no sea escaneado.")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text       = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"attention_paper.pdf → {len(attention_text):,} chars")
print(f"rag_paper.pdf       → {len(rag_text):,} chars")
print(f"\nPrimeros 200 chars del paper de Attention:\n{attention_text[:200]}...")

attention_paper.pdf → 40,174 chars
rag_paper.pdf       → 69,121 chars

Primeros 200 chars del paper de Attention:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShaz...


---
## Parte 2 — Convertir a Documents y chunkear

LangChain trabaja con objetos `Document` que tienen:
- `page_content` — el texto del chunk
- `metadata`     — dict con información del origen (título, fuente, etc.)

**Tu tarea:** completar `pdf_to_documents()` que convierte el texto de un PDF
en una lista de `Document` usando `RecursiveCharacterTextSplitter`.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ── TODO 2 ───────────────────────────────────────────────────
# Completa la función pdf_to_documents():
# - Crea un RecursiveCharacterTextSplitter con chunk_size=600, chunk_overlap=60
# - Usa text_splitter.split_text(text) para obtener los chunks
# - Por cada chunk crea un Document con:
#     page_content = chunk
#     metadata = {"title": title, "source": source, "chunk_idx": i}
# - Retorna la lista de Documents
# - Descarta chunks con menos de 30 caracteres

def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=60,
        separators=["\n\n", "\n", ".", " "],
    )
    chunks = text_splitter.split_text(text)
    docs = []
    for i, chunk in enumerate(chunks):
        if len(chunk.strip()) < 30:
            continue
        docs.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_idx": i},
        ))
    return docs


# ── Aplicar a los dos papers ──────────────────────────────────
attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)

all_docs = attention_docs + rag_docs

print(f"attention_paper → {len(attention_docs)} chunks")
print(f"rag_paper       → {len(rag_docs)} chunks")
print(f"Total           → {len(all_docs)} chunks")
print(f"\nEjemplo de Document:")
print(f"  page_content: {all_docs[0].page_content[:100]}...")
print(f"  metadata:     {all_docs[0].metadata}")

attention_paper → 80 chunks
rag_paper       → 134 chunks
Total           → 214 chunks

Ejemplo de Document:
  page_content: Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthi...
  metadata:     {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_idx': 0}


---
## Parte 3 — Embeddings e índice Qdrant

Ahora indexamos los Documents en Qdrant usando Gemini para los embeddings.

**Tu tarea:** inicializar el `GoogleGenerativeAIEmbeddings` y crear el
`QdrantVectorStore` indexando todos los documentos.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "papers_rag"
EMBEDDING_DIM = 768   # gemini-embedding-001 con output_dimensionality=768

# ── TODO 3a ──────────────────────────────────────────────────
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
    output_dimensionality=EMBEDDING_DIM,
)

# ── TODO 3b ──────────────────────────────────────────────────
qdrant_client = QdrantClient(":memory:")
if qdrant_client.collection_exists(COLLECTION):
    qdrant_client.delete_collection(COLLECTION)
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

# ── TODO 3c ──────────────────────────────────────────────────
vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
# El free tier de Gemini permite 100 embeddings/min: se indexa por lotes con pausa
import time
LOTE = 90
for i in range(0, len(all_docs), LOTE):
    vectorstore.add_documents(all_docs[i:i + LOTE])
    print(f"  lote {i // LOTE + 1}: {min(i + LOTE, len(all_docs))}/{len(all_docs)} chunks indexados")
    if i + LOTE < len(all_docs):
        time.sleep(65)

info = qdrant_client.get_collection(COLLECTION)
print(f"Indexados {len(all_docs)} chunks en Qdrant (puntos en la colección: {info.points_count})")

  lote 1: 90/214 chunks indexados


  lote 2: 180/214 chunks indexados


  lote 3: 214/214 chunks indexados
Indexados 214 chunks en Qdrant (puntos en la colección: 214)


---
## Parte 4 — Pipeline RAG con LCEL

Con el índice listo, construimos el pipeline RAG usando el operador `|` de LangChain.

**Tu tarea:** construir el pipeline completo conectando retriever → prompt → LLM.

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_core.rate_limiters import InMemoryRateLimiter

# ── TODO 4a ──────────────────────────────────────────────────
# Free tier: 10 peticiones/min → el limitador espacia las llamadas (~8/min)
rate_limiter = InMemoryRateLimiter(requests_per_second=0.13, check_every_n_seconds=0.5, max_bucket_size=1)
llm = ChatGoogleGenerativeAI(
    rate_limiter=rate_limiter,
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.1,
)

# ── TODO 4b ──────────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.3},
)

# ── TODO 4c ──────────────────────────────────────────────────
def format_docs(docs: List[Document]) -> str:
    """Convierte lista de Documents a string de contexto para el prompt."""
    if not docs:
        return "(no se recuperó ningún fragmento relevante)"
    return "\n\n---\n\n".join(
        f"[{d.metadata['title']} · chunk {d.metadata['chunk_idx']}]\n{d.page_content}" for d in docs
    )

# ── TODO 4d ──────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_template(
    """You are a research assistant that answers questions about two AI papers.
Answer ONLY with information contained in the context below. Cite the paper title
in brackets for each fact you use. If the context does not contain the answer,
reply exactly: "I don't know based on the indexed papers." Do not use prior knowledge.

Context:
{context}

Question: {question}

Answer:"""
)

# ── TODO 4e ──────────────────────────────────────────────────
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ── Prueba rápida ─────────────────────────────────────────────
test_answer = rag_chain.invoke("What is the Transformer architecture?")
print(f"Prueba:\n{test_answer[:300]}...")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Prueba:
The Transformer is a new simple network architecture that relies solely on attention mechanisms, dispensing with recurrence and convolutions entirely. [Attention Is All You Need · chunk 1] It follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both...


---
## Parte 5 — Quiz: preguntas sobre los papers

Responde estas preguntas con tu sistema RAG.
Las respuestas correctas están al final — compara con lo que genera el sistema.

Primero corre cada celda y observa la respuesta del RAG.
Luego verifica contra el ground truth.

In [7]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

Q: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
A: The big Transformer model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task. [Attention Is All You Need · chunk 39]


<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [8]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

Q: How does RAG combine parametric and non-parametric memory?
A: RAG models combine pre-trained parametric and non-parametric memory for language generation. [Retrieval-Augmented Generation · chunk 1]


<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [9]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

Q: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
A: The scaling factor $\sqrt{1/d_k}$ is used to counteract the effect of large dot products in Scaled Dot-Product Attention. For large values of $d_k$, the dot products can grow large in magnitude, pushing the softmax function into regions with extremely small gradients. Scaling the dot products by $\sqrt{1/d_k}$ helps to prevent this. [Attention Is All You Need · chunk 17]


<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [10]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

Q: What are the two RAG model variants proposed in the paper and how do they differ?
A: The paper proposes two models: RAG-Sequence and RAG-Token. In RAG-Sequence, the model uses the same document to predict each target token. In contrast, RAG-Token can predict each target token based on a different document, allowing it to choose content from several documents when producing an answer [Retrieval-Augmented Generation · chunk 16]. The RAG-Token model draws a different latent document for each target token and marginalizes accordingly, while RAG-Sequence assumes the same document is responsible for all tokens [Retrieval-Augmented Generation · chunk 10].


<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [11]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

Q: How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?
A: The base Transformer model employs 8 parallel attention layers, or heads. For each of these, the dimensions for key and value are $d_k = d_v = d_{model}/h = 64$. [Attention Is All You Need · chunk 19, Attention Is All You Need · chunk 20]


<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta cross-paper

Esta pregunta requiere información de **ambos** papers a la vez.
Observa si el RAG recupera chunks de los dos documentos.

In [12]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

Q: The RAG paper uses FAISS for fast retrieval with an HNSW approximation. The Attention paper proposes the Transformer architecture. What seq2seq model does RAG use as its generator component, and is it based on the Transformer?

A: The RAG model uses BART as its seq2seq model component. [Retrieval-Augmented Generation · chunk 10]

Fuentes recuperadas:
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf


<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## Prueba adicional — pregunta fuera del alcance de los papers

Para responder la primera pregunta de la reflexión, se le hace al sistema una pregunta cuya respuesta **no está** en ninguno de los dos papers y se observa si admite que no sabe o si inventa.

In [13]:
q_fuera = "What was the population of Medellín, Colombia in 2020?"
docs_fuera = retriever.invoke(q_fuera)
answer_fuera = rag_chain.invoke(q_fuera)
print(f"Q: {q_fuera}")
print(f"Chunks recuperados (score >= 0.3): {len(docs_fuera)}")
for d in docs_fuera:
    print(f"  [{d.metadata['title']}] chunk {d.metadata['chunk_idx']}: {d.page_content[:80]!r}")
print(f"\nA: {answer_fuera}")

Q: What was the population of Medellín, Colombia in 2020?
Chunks recuperados (score >= 0.3): 4
  [Retrieval-Augmented Generation] chunk 80: 'pages209–220,Vancouver,Canada,July2017.AssociationforComputationalLinguistics. d'
  [Retrieval-Augmented Generation] chunk 93: 'Chris Alberti, Danielle Epstein, Illia Polosukhin, Matthew Kelcey, Jacob Devlin,'
  [Retrieval-Augmented Generation] chunk 82: 'URLhttps://www.aclweb.org/anthology/N19-1423.\n[9] EmilyDinan,StephenRoller,KurtS'
  [Retrieval-Augmented Generation] chunk 78: 'fromQuestion-AnswerPairs. InProceedingsofthe2013ConferenceonEmpiricalMethods\ninN'

A: I don't know based on the indexed papers.


In [14]:
# Scores reales de similitud para la pregunta bonus y la pregunta fuera de alcance
for q in [q_bonus, q_fuera]:
    print(f"\nQ: {q[:70]}...")
    for d, s in vectorstore.similarity_search_with_score(q, k=6):
        print(f"  {s:.3f}  [{d.metadata['title']}] chunk {d.metadata['chunk_idx']}")


Q: The RAG paper uses FAISS for fast retrieval with an HNSW approximation...


  0.771  [Retrieval-Augmented Generation] chunk 10
  0.770  [Retrieval-Augmented Generation] chunk 2
  0.767  [Retrieval-Augmented Generation] chunk 17
  0.756  [Retrieval-Augmented Generation] chunk 24
  0.755  [Retrieval-Augmented Generation] chunk 19
  0.752  [Retrieval-Augmented Generation] chunk 18

Q: What was the population of Medellín, Colombia in 2020?...


  0.523  [Retrieval-Augmented Generation] chunk 80
  0.508  [Retrieval-Augmented Generation] chunk 93
  0.507  [Retrieval-Augmented Generation] chunk 82
  0.507  [Retrieval-Augmented Generation] chunk 78
  0.507  [Retrieval-Augmented Generation] chunk 87
  0.505  [Retrieval-Augmented Generation] chunk 92


---
## Resultados del quiz vs. ground truth

| # | Pregunta | Respuesta del RAG | Veredicto |
|---|---|---|---|
| 1 | BLEU del Transformer big (EN→DE) | 28.4 (chunk 39) | ✅ Exacta |
| 2 | Memoria paramétrica + no paramétrica | "combina memoria paramétrica y no paramétrica pre-entrenadas" (solo chunk 1) | 🟡 Correcta pero superficial: no menciona BART, DPR ni el índice de Wikipedia |
| 3 | Propósito de 1/√dk | Evita productos punto grandes que llevan el softmax a gradientes minúsculos (chunk 17) | ✅ Correcta |
| 4 | RAG-Sequence vs RAG-Token | Mismo documento para toda la secuencia vs. documento distinto por token (chunks 10 y 16) | ✅ Correcta y completa |
| 5 | Cabezas y dimensiones del Transformer base | h = 8, dk = dv = 64 (chunks 19–20) | ✅ Exacta |
| Bonus | Generador de RAG y si es un Transformer | "BART" (chunk 10) | 🟡 Parcial: no conecta BART con el Transformer, porque solo recuperó chunks del paper de RAG |
| Extra | Población de Medellín (fuera de alcance) | "I don't know based on the indexed papers." | ✅ No alucinó |

**Resultado: 4/5 correctas + 1 superficial; bonus parcial; 0 alucinaciones.**

---
## Reflexión final

Responde estas preguntas basándote en lo que observaste durante el workshop:

1. **¿Qué pasó cuando el RAG no encontró información relevante?**
   ¿Respondió correctamente que no sabía, o inventó algo?

2. **¿Qué chunks recuperó para la pregunta bonus?**
   ¿Fueron de ambos papers o solo de uno?

3. **¿Qué cambiarías del pipeline** para mejorar la calidad de las respuestas?
   Piensa en: chunk_size, score_threshold, k, el prompt.

---
## Reflexión final — respuestas

**1. ¿Qué pasó cuando el RAG no encontró información relevante?**
Dijo que no sabía, que es lo que tenía que hacer. Pero ojo con por qué: el *retriever* **sí** le pasó 4 chunks para la pregunta de Medellín, todos de referencias bibliográficas del paper de RAG, con similitudes de 0.50–0.52. Con los embeddings de Gemini, hasta un texto sin ninguna relación supera cómodamente el umbral de 0.3, así que el umbral no filtró nada. Lo que evitó la alucinación fue el **prompt**, que obliga a responder solo con el contexto y trae una frase exacta para cuando no hay respuesta. El filtro del retriever no sirvió; la disciplina del prompt sí.

**2. ¿Qué chunks recuperó para la pregunta bonus?**
Solo del paper de RAG (chunks 10, 2, 17 y 24, con similitud de 0.75–0.77). La pregunta menciona FAISS, HNSW y "RAG", y esas palabras la acercan a ese paper. Ningún chunk del paper de *Attention* entró en el top-4. Por eso la respuesta se quedó en "BART" y no llegó al "es un Transformer": el sistema es tan bueno como lo que recupera, y aquí recuperó la mitad de la historia.

**3. ¿Qué cambiaría del pipeline?**
- **Umbral calibrado, no a ojo.** Con Gemini, las preguntas irrelevantes quedan alrededor de 0.51 y las relevantes por encima de 0.74. Un `score_threshold` de ~0.65 cortaría la basura sin perder los aciertos.
- **Más diversidad en la recuperación.** Usaría MMR (`search_type="mmr"`) o k=6 con un mínimo por documento, para que las preguntas *cross-paper* traigan evidencia de ambos papers.
- **Limpiar el PDF antes de chunkear.** MarkItDown pegó palabras sin espacios (`Providedproperattributionisprovided…`), y eso degrada tanto los embeddings como las citas. Un paso de normalización, o extraer con PyMuPDF, mejora la materia prima.
- **Filtrar las referencias bibliográficas.** Esos chunks casi no aportan y son justo los que aparecen como falsos positivos. Cortaría el texto en la sección *References*.
- **Chunks un poco más grandes (~900) para preguntas conceptuales como la P2.** Con 600 caracteres, la respuesta quedó en una sola frase del abstract.

En resumen, el modelo no fue el cuello de botella; lo fueron la calidad de los datos y el retriever. Cambiar el LLM por uno más caro habría costado más sin arreglar nada de lo que realmente falló.